# ColPali (ColQwen2) vs OCR retrieval benchmark

Reproduces every number in the README on a free Colab T4. Runtime: **GPU (T4)**.
Each subset has its own run cell; artifacts (OCR cache, embeddings) persist on Drive so a disconnect resumes cleanly.

**Edit `REPO_URL` below** to point at your own GitHub copy of this repo.

In [ ]:
REPO_URL = "https://github.com/pram-pram/colpali-vs-ocr.git"

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Switch the runtime to a GPU (Runtime > Change runtime type > T4)."
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone $REPO_URL colpali-vs-ocr
%cd colpali-vs-ocr
!pip install -q -e ".[ocr,dense,vision]" "colpali-engine>=0.3.4,<0.4"

In [ ]:
# `pip install -e .` only takes effect in a fresh interpreter; put the src layout on
# this running kernel's path so `from cvo ...` works below.
import sys; sys.path.insert(0, "src")

from importlib.metadata import version
for pkg in ["torch", "transformers", "colpali-engine", "python-doctr", "sentence-transformers"]:
    print(pkg, version(pkg))

## Artifacts location
Mounting Drive keeps OCR caches, embeddings and results JSONs across sessions. Without Drive, they go to the local `artifacts`, `results` and `results_smoke` folders and are lost on disconnect.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
ARTIFACTS = (
    "/content/drive/MyDrive/colpali-vs-ocr/artifacts"
    if os.path.isdir("/content/drive/MyDrive")
    else "artifacts"
)
DRIVE_ROOT = "/content/drive/MyDrive/colpali-vs-ocr"
RESULTS = f"{DRIVE_ROOT}/results" if os.path.isdir("/content/drive/MyDrive") else "results"
RESULTS_SMOKE = f"{DRIVE_ROOT}/results_smoke" if os.path.isdir("/content/drive/MyDrive") else "results_smoke"
os.environ["ARTIFACTS"] = ARTIFACTS
print("ARTIFACTS =", ARTIFACTS)
print("RESULTS =", RESULTS)
print("RESULTS_SMOKE =", RESULTS_SMOKE)

## Tests (includes the GPU ColQwen test)

In [ ]:
!python -m pytest -v

## Smoke run (50 pages, all arms)
Check this works before spending hours on the full runs.

In [ ]:
!python -m cvo.run --subset tatdqa --arms bm25,dense,colqwen --limit 50 --artifacts $ARTIFACTS --results $RESULTS_SMOKE

## Full runs
One cell per subset. Re-running a cell resumes from cached OCR pages / embedding shards.

In [ ]:
!python -m cvo.run --subset shift --arms bm25,dense,colqwen --artifacts $ARTIFACTS --results $RESULTS

In [ ]:
!python -m cvo.run --subset tatdqa --arms bm25,dense,colqwen --artifacts $ARTIFACTS --results $RESULTS

In [ ]:
!python -m cvo.run --subset infovqa --arms bm25,dense,colqwen --artifacts $ARTIFACTS --results $RESULTS

In [ ]:
!python -m cvo.run --subset arxivqa --arms bm25,dense,colqwen --artifacts $ARTIFACTS --results $RESULTS

## Report: table and charts

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display
from cvo.report import load_results, plot_all, summary_table

results = load_results(Path(RESULTS))
display(Markdown(summary_table(results)))
for path in plot_all(results, Path(RESULTS) / "plots"):
    display(Image(filename=str(path)))

## Failure gallery
Queries where ColQwen2 is right at rank 1 and BM25 has nothing in its top 5, and the reverse, with the top-1 page each arm returned.

In [ ]:
import matplotlib.pyplot as plt
from cvo.data import load_subset
from cvo.report import select_gallery

GALLERY_SUBSET = "tatdqa"  # change to inspect another subset
pages, queries = load_subset(GALLERY_SUBSET)
page_by_id = {p.id: p for p in pages}  # decoded only when shown
query_by_id = {q.id: q for q in queries}
relevant = {q.id: set(q.relevant_page_ids) for q in queries}

missing = [a for a in ("colqwen", "bm25") if a not in results.get(GALLERY_SUBSET, {})]
if missing:
    print(f"No results for {missing} on {GALLERY_SUBSET!r}; run those arms first.")
    gallery = {}
else:
    colqwen = results[GALLERY_SUBSET]["colqwen"]
    bm25 = results[GALLERY_SUBSET]["bm25"]
    gallery = select_gallery(colqwen, bm25, relevant, n=2)
    if not any(gallery.values()):
        print("No queries where one arm is right at rank 1 and the other misses its top 5.")

def top1(result, qid):
    ranked = result["per_query"].get(qid) or []
    return ranked[0][0] if ranked else None  # BM25 returns [] when no query term matches

def show(qid):
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    panels = [
        ("relevant page", sorted(relevant[qid])[0]),
        ("colqwen top-1", top1(colqwen, qid)),
        ("bm25 top-1", top1(bm25, qid)),
    ]
    for ax, (title, page_id) in zip(axes, panels):
        if page_id is None:
            ax.text(0.5, 0.5, "(no result)", ha="center", va="center")
            ax.set_title(title, fontsize=8)
        else:
            ax.imshow(page_by_id[page_id].pil())
            ax.set_title(f"{title}\n{page_id}", fontsize=8)
        ax.axis("off")
    fig.suptitle(query_by_id[qid].text, wrap=True)
    plt.show()

for label, qids in gallery.items():
    print(label, qids)
    for qid in qids:
        show(qid)

## Save results into the repo
Copies the results JSONs and plots into this checkout's `results/` folder. Commit and push `results/` from here (or download it) so the README numbers are reproducible from the repo.

In [ ]:
import shutil
if os.path.abspath(RESULTS) != os.path.abspath("results"):
    shutil.copytree(RESULTS, "results", dirs_exist_ok=True)
print(sorted(str(p) for p in Path("results").rglob("*.json")))